### Preprocessing

In [ ]:
import pandas as pd

# 1. Load the raw CSV
file_path = "MICRO DATA - 2020 - Upasana.csv"  # adjust path as needed
df = pd.read_csv(file_path)

# 2. Drop all unnamed/junk columns
df = df.loc[:, ~df.columns.str.contains(r'^Unnamed')]

# 3. Extract numeric part of Age and convert to nullable integer
df['Age'] = df['Age'].str.extract(r'(\d+)').astype('Int64')

# 4. Parse Date (MM/DD/YYYY) and reformat to ISO 8601 (YYYY-MM-DD)
df['Date'] = (
    pd.to_datetime(df['Date'], format='%m/%d/%Y', errors='coerce')
      .dt.strftime('%Y-%m-%d')
)

# 5. Trim whitespace and lowercase all text fields
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip().str.lower()

# 6. Replace empty strings with NULL (pd.NA)
df = df.replace({'': pd.NA})

# Now df is preprocessed and ready for PostgreSQL ingestion
df.to_csv("preprocessed_data.csv", index=False)

### OpenAI Authentication

In [1]:
from getpass import getpass
import openai

api_key = getpass("Enter your OpenAI API key: ")
client = openai.OpenAI(api_key=api_key)


### Simple Streamlit Deployment

In [4]:
import pandas as pd
import streamlit as st
import random

# ─── 1. Load your preprocessed CSV into a DataFrame ────────────────────────────
df = pd.read_csv("C:/Rishika_PC/Projects/Incubate/incubate_2025/preprocessed_data.csv")

# ─── 1a. Derive a clean `patient_id` column from `UMR No.` ──────────────────────
df['patient_id'] = (
    df['UMR No.']                 # use the original UMR No. column
      .astype(str)
      .str.replace(r'\.0$', '', regex=True)
      .str.strip()
)

# ─── 2. Simulate geo_zones if not already present ─────────────────────────────
zones = ['Zone A', 'Zone B', 'Zone C']
if 'geo_zone' not in df.columns:
    df['geo_zone'] = [random.choice(zones) for _ in range(len(df))]

# ─── 3. Streamlit UI ───────────────────────────────────────────────────────────
st.title("Antibiotic Resistance Checker")

patient_id = st.text_input("Enter Patient UMR No.")
antibiotic  = st.text_input("Enter Antibiotic Name")

# ─── 4. Helper: match antibiotic input to column ───────────────────────────────
def get_antibiotic_col(name: str):
    for col in df.columns:
        if col.lower() == name.lower():
            return col
    return None

# ─── 5. Core logic ─────────────────────────────────────────────────────────────
def check_patient_resistance(pid: str, ab_name: str):
    col = get_antibiotic_col(ab_name)
    if not col:
        return None
    sub = df[df['patient_id'] == pid]
    return (sub[col] == 'r').any()

def get_patient_zone(pid: str):
    try:
        return df[df['patient_id'] == pid]['geo_zone'].iloc[0]
    except IndexError:
        return None

def check_region_resistance(zone: str, ab_name: str, threshold: int = 5):
    col = get_antibiotic_col(ab_name)
    if not col:
        return None
    region = df[df['geo_zone'] == zone]
    return (region[col] == 'r').sum() >= threshold

def generate_alert_message(patient_flag: bool, region_flag: bool) -> str:
    reasons = []
    if patient_flag:
        reasons.append("patient has a known resistance")
    if region_flag:
        reasons.append("this region shows significant resistance")
    reason_str = " and ".join(reasons)
    prompt = (
        f"A doctor is considering prescribing an antibiotic, "
        f"but the {reason_str}. Generate a formal medical alert."
    )
    resp = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role":"user","content":prompt}]
    )
    return resp.choices[0].message.content

# ─── 6. Button handler ────────────────────────────────────────────────────────
if st.button("Check Resistance"):
    if not patient_id or not antibiotic:
        st.warning("Please enter both Patient UMR No. and Antibiotic name.")
    else:
        p_flag = check_patient_resistance(patient_id, antibiotic)
        if p_flag is None:
            st.error("Antibiotic not found in the dataset.")
        else:
            zone = get_patient_zone(patient_id)
            if zone is None:
                st.error("Patient not found.")
            else:
                r_flag = check_region_resistance(zone, antibiotic)
                if r_flag is None:
                    st.error("Antibiotic not found in the dataset.")
                elif p_flag or r_flag:
                    alert = generate_alert_message(p_flag, r_flag)
                    st.error(alert)
                else:
                    st.success("No resistance found. Antibiotic is safe to proceed.")

2025-07-13 18:18:02.898 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-13 18:18:02.936 
  command:

    streamlit run c:\Rishika_PC\Projects\Incubate\incubate_2025\myenv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-07-13 18:18:02.937 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-13 18:18:02.937 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-13 18:18:02.938 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-13 18:18:02.939 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-13 18:18:02.939 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-13 18:18:02.